In [0]:
## TODO need widgets to pass variables to/from nbs/tasks
# Update paths 

In [0]:
# Currently in this nb -- to decide if quick instance inference to stay here or in next nb (Model Inference)

# `Transfer Learning`

### Overview of YOLO Model and Recent Updates

- #### YOLO Model
YOLO (You Only Look Once) is a state-of-the-art, real-time object detection system. It frames object detection as a single regression problem, straight from image pixels to bounding box coordinates and class probabilities. This approach allows YOLO to achieve high accuracy and speed, making it suitable for real-time applications.

- #### Recent Updates: [Instance Segmentation](https://docs.ultralytics.com/tasks/segment/#models)
Recent updates to the YOLO model have introduced capabilities for instance segmentation. Instance segmentation not only detects objects but also delineates the exact shape of each object, providing pixel-level masks. This is particularly useful in applications requiring precise object boundaries, such as medical imaging and autonomous driving.

- #### Transfer Learning
Transfer learning involves taking a pre-trained model and fine-tuning it on a new dataset. This approach leverages the knowledge gained from a large dataset and applies it to a specific task, reducing the need for extensive computational resources and training time. In the context of YOLO, transfer learning allows us to adapt the model to new object classes or domains with limited data.

- #### Note on architecture and segmentation implementation 
   - **YOLO Architecture**: The YOLO architecture consists of convolutional layers followed by fully connected layers, designed to predict bounding boxes and class probabilities directly from the input image.
   - **Instance Segmentation**: The recent YOLO models incorporate segmentation heads that output pixel-level masks for each detected object, enhancing the model's ability to perform instance segmentation.

---    


For more detailed information, refer to the original YOLO papers and the latest research on instance segmentation [`hidden` in the next cell].

<!-- For more detailed information, refer to the original YOLO papers and the latest research on instance segmentation. -->

##### YOLO Refs:
1. **[YOLOv1 Paper](https://arxiv.org/abs/1506.02640): You Only Look Once: Unified, Real-Time Object Detection** (Joseph Redmon, Santosh Divvala, Ross Girshick, Ali Farhadi)
2. **[YOLOv2 Paper](https://arxiv.org/abs/1612.08242): YOLO9000: Better, Faster, Stronger** (Joseph Redmon, Ali Farhadi)
3. **[YOLOv3 Paper](https://arxiv.org/abs/1804.02767): An Incremental Improvement** (Joseph Redmon, Ali Farhadi)
4. **[YOLOv4 Paper](https://arxiv.org/abs/2004.10934): Optimal Speed and Accuracy of Object Detection** (Alexey Bochkovskiy, Chien-Yao Wang, Hong-Yuan Mark Liao)
5. **[YOLOv5 GitHub](https://github.com/ultralytics/yolov5): PyTorch Implementation** (YOLOv5 is not an official paper but an implementation by Ultralytics)
6. **[YOLOv6 Paper](https://arxiv.org/abs/2209.02976): A Single-Stage Object Detection Framework for Industrial Applications** (Meituan)
7. **[YOLOv7 Paper](https://arxiv.org/abs/2207.02696): Trainable bag-of-freebies sets new state-of-the-art for real-time object detectors** (Chien-Yao Wang, Alexey Bochkovskiy, Hong-Yuan Mark Liao)
8. **[YOLOv8 GitHub](https://github.com/ultralytics/ultralytics): Ultralytics YOLOv8**
9. **[Ultralytics YOLOv11](https://docs.ultralytics.com/models/yolo11/#key-features)** (Latest version by Ultralytics, focusing on improved performance and ease of use)

In [0]:
!pip install -q ultralytics pycocotools scikit-learn matplotlib 

%restart_python

In [0]:
import os
import shutil

import json

import random
import numpy as np

import time
import datetime

import matplotlib.pyplot as plt

from IPython.display import Image
from sklearn.model_selection import train_test_split

import torch
import torch.utils.data
from torch import nn
import torchvision
from torchvision import transforms as T

from ultralytics import YOLO

from pycocotools import mask as coco_mask
from pycocotools.coco import COCO

In [0]:
CATALOG_NAME = "mmt"
SCHEMA_NAME = "cv"

VOLUME_PATH = f"/Volumes/{CATALOG_NAME}/{SCHEMA_NAME}"

# DATA_DIR = f"{VOLUME_PATH}/data"
PROJECTS_DIR = f"{VOLUME_PATH}/projects"

PROJECT_PATH = f"{PROJECTS_DIR}/NuInsSeg"

PROJECT_RAWDATA_DIR = f"{PROJECT_PATH}/raw_data"

# ZIPFILE_PATH = f"{PROJECT_RAWDATA_DIR}/nuinsseg.zip"

YOLO_DATA_DIR = f"{PROJECT_PATH}/yolo_dataset" # can update this to change the path to your own data 

# Get the current working directory
nb_context = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
current_path = f"/Workspace{nb_context}"
# print(f"Current path: {current_path}")
WS_PROJ_DIR = '/'.join(current_path.split('/')[:-1]) 

WORKSPACE_PATH = dbutils.notebook.entry_point.getDbutils().notebook().getContext().tags().get("user").get()
USER_WORKSPACE_PATH = f"/Users/{WORKSPACE_PATH}"

# project_name = "yolo_MedCellTypes_InstanceSeg"
# experiment_name = USER_WORKSPACE_PATH + "mlflow_experiments/yolo/{project_name}"
# print(f"Setting experiment name to be {experiment_name}")

--------------------------

#### We will first illustrate the `Default` local workspace paths used by Ultralytics.

Subsequently, we will re-define these default paths later to illustrate how one would use the preprocessed image datasets ingested and written to UC Vols. and organize the generated assets in model training 

In [0]:
from ultralytics import settings

# View all settings
print(settings)

# datasets_dir	'/path/to/datasets'	str	The directory where the datasets are stored
# weights_dir	'/path/to/weights'	str	The directory where the model weights are stored
# runs_dir	'/path/to/runs'	str	The directory where the experiment runs are stored

In [0]:
YOLO_DATA_DIR, WS_PROJ_DIR, USER_WORKSPACE_PATH

In [0]:
# Copy all files except for data.yaml from YOLO_DATA_DIR to WS_PROJ_DIR
# !rsync -av --exclude='data.yaml' {YOLO_DATA_DIR}/ {WS_PROJ_DIR}/datasets/

#### Run a 'quick' model training to illustrate where data/folder paths are found 'by default' 

In [0]:
yolo_default = False
# yolo_default = True

In [0]:
if yolo_default:

    ## "DEFAULT" Quick Start 

    import torch.distributed as dist
    from ultralytics import YOLO

    # Initialize the process group
    if not dist.is_initialized():
        dist.init_process_group(backend='nccl') ## required for cuda 

    try:
        # Transfer the weights from a pretrained model (recommended for training)
        model = YOLO("yolo11n-seg.pt") 
        ## also this will be loaded to the path; if not specified this will be loaded to directory where model training code nb resides WS_PROJ_DIR 

        results = model.train(                          
                            data=f"{WS_PROJ_DIR}/data.yaml", ## default settings, YOLO assumes a "datasets" dir in directory where model training code nb resides WS_PROJ_DIR
                            # data = YOLO_DATA_DIR, ## we will subsequently update this to our YOLO data.yaml file & data path in UC
                            epochs=100,
                            patience=0,  # setting patience=0 to disable early stopping
                            batch=3,
                            imgsz=1024,
                            optimizer="adam", #"sgd",
                            )
        
    finally:
        # Destroy the process group
        dist.destroy_process_group()

In [0]:
# %load_ext tensorboard (MLflow*)
# %tensorboard --logdir runs/segment/train

In [0]:
If yolo_default():
  Image("runs/segment/train/results.png")   

### `[MOVE to Inference Section] TBD?`

### Quick Inference with YOLO framework + best weights 

In [0]:
from ultralytics import YOLO
import matplotlib.pyplot as plt
import cv2
import os
import random

# Define the project directory
test_data_path = f"{WS_PROJ_DIR}/datasets/test/images"

# Load the trained YOLO model
model = YOLO("./runs/segment/train/weights/best.pt")

# Load test data
print(test_data_path)

# Check if the directory exists
if not os.path.exists(test_data_path):
    raise FileNotFoundError(f"The directory {test_data_path} does not exist.")

test_images = [os.path.join(test_data_path, img) for img in os.listdir(test_data_path) if img.endswith('.png')]

# Randomly select 25 images
selected_images = random.sample(test_images, 25)

# Resize images to a smaller size
resized_images = []
for img_path in selected_images:
    img = cv2.imread(img_path)
    resized_img = cv2.resize(img, (640, 640))  # Resize to 640x640 or any desired size
    resized_images.append(resized_img)

# Batch Predict using the loaded model
results = model.predict(resized_images)

# Plot 5x5 grid of test images with predictions
fig, axes = plt.subplots(5, 5, figsize=(10, 10))
axes = axes.flatten()
for img, result, ax in zip(resized_images, results, axes):
    for box in result.boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0])  # Flatten the list before mapping to int
        cv2.rectangle(img, (x1, y1), (x2, y2), (255, 255, 255), 2)
        # cv2.putText(img, f"{box.cls}", (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255, 255, 255), 2)
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    ax.axis('off')
plt.tight_layout()
plt.show()

## speed depends on network

Using the `Default` Ultralytics `model.predict()` on a random sample of test images illustrates how efficient the inference can be on YOLO-formatted test images. 

--------------------------

### CHALLENGE(s) with `Default` Ultralytics `settings` 

Minimal MLflow integration -- mainly model training and validation tracking 

- [a] Not Logged 
  - experiment_id: ?
  - run_id : ?
  - experiment_name: when not set --> "Ultralytics"
    - 🏃 View run train at: https://e2-demo-field-eng.cloud.databricks.com/ml/experiments/971429592633706/runs/1f43edc75344493689900f59cdf9e4bc
    - 🧪 View experiment at: https://e2-demo-field-eng.cloud.databricks.com/ml/experiments/971429592633706

- [b] "Clogging" up Workspace folder with default files written there --> ideally to update in /write to UC
  - where `yolo_dataset` is written
  - organize `runs` (currently defaulting to workspace nb directory path)
  - `results` currently saved to `./runs/segment/train{#}`

- [c] Not Monitored / Leveraged
  - Not Tracking System metrics 
  - Not Leveraging ALL Compute GPUs 
  - Not able to log trained model easily

--------------------------

### Addressing [a] & [b]: Add Some Organization to Specify & Organize Training Paths 
- Store mlflow run artifacts + results updates to Unity Catalog Volumes

In [0]:
settings

In [0]:
# Update setting
settings.update({"datasets_dir": f"{YOLO_DATA_DIR}",
                ## UC Volumes do not allow appending to the path, so these specifications will not be 'used' even if they were to be specified
                #  "weights_dir": f"{PROJECT_PATH}/training_runs/{runs}/segment/train/weights",  
                #  "runs_dir": f"{PROJECT_PATH}/training_runs"
                })

# View all settings
print(settings) 

In [0]:
# import os
# Config project structure directory under UC
# project_location = '/Volumes/yyang/computer_vision/Nuclei_Instance/'

project_name = "yolo_MedCellTypes_InstanceSeg"
experiment_name = USER_WORKSPACE_PATH + "mlflow_experiments/yolo/{project_name}"
# print(f"Setting experiment name to be {experiment_name}")


DATA_PROJECT_training_runs = f'{PROJECT_PATH}/training_runs/'
os.makedirs(DATA_PROJECT_training_runs, exist_ok=True)

# os.makedirs(f'{PROJECT_PATH}/data/', exist_ok=True)

DATA_PROJECT_yolo_model = f'{PROJECT_PATH}/yolo_model/'
os.makedirs(DATA_PROJECT_yolo_model, exist_ok=True)

# for cache related to ultralytics
os.environ['ULTRALYTICS_CACHE_DIR'] = DATA_PROJECT_yolo_model


# volume folder in UC.
DATA_PROJECT_training_results = f'{PROJECT_PATH}/training_results/'
volume_project_location = DATA_PROJECT_training_results
os.makedirs(volume_project_location, exist_ok=True)

# or more traditional way, setup folder under DBFS.
# dbfs_project_location = '/dbfs/tmp/cv_project_location/Nuclei_Instance/'
# os.makedirs(dbfs_project_location, exist_ok=True)

# # ephemeral /tmp/ project location on VM, good for Appending operation during training.
tmp_project_location = "/tmp/mmt_nuinsseg_yolo_training_results/"
os.makedirs(tmp_project_location, exist_ok=True)

--------------------------

### TRAINING with updated paths  

In [0]:

def set_seeds():
    # fix random seeds
    SEED_VALUE = 42

    random.seed(SEED_VALUE)
    np.random.seed(SEED_VALUE)
    torch.manual_seed(SEED_VALUE)
    
    if torch.cuda.is_available():
        torch.cuda.manual_seed(SEED_VALUE)
        torch.cuda.manual_seed_all(SEED_VALUE)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = True
            
set_seeds()

In [0]:
# Import the YOLO class from the ultralytics library
from ultralytics import YOLO

import torch
import torch.distributed as dist
import mlflow
import os

mlflow.set_tracking_uri("databricks")
mlflow.set_registry_uri("databricks-uc")
mlflow.end_run()

# Enable MLflow autologging
mlflow.autolog()

# Initialize the process group
if not dist.is_initialized():
    dist.init_process_group(backend='nccl')  # required for cuda

# Define the experiment name
experiment_name = USER_WORKSPACE_PATH + "/mlflow_experiments/yolo/yolo_MedCellTypes_InstanceSeg"
print(f"Setting experiment name: {experiment_name}")

# Create the experiment if it does not exist
experiment = mlflow.get_experiment_by_name(experiment_name)
if experiment is None:
    ARTIFACT_PATH = f"dbfs:{PROJECT_PATH}/artifact"
    print(f"Creating experiment ARTIFACT_PATH to be {ARTIFACT_PATH}")
    experiment_id = mlflow.create_experiment(name=experiment_name, artifact_location=ARTIFACT_PATH)
else:
    experiment_id = experiment.experiment_id

try:
    with mlflow.start_run(log_system_metrics=True, experiment_id=experiment_id) as run:
        model = YOLO(f"{PROJECT_PATH}/yolo_model/yolo11n-seg.pt")       

        # Train the model using the data.yaml file
        results = model.train(
            data=os.path.join(f"{YOLO_DATA_DIR}", "data.yaml"),
            epochs=100, 
            # patience=0,  # setting patience=0 to disable early stopping
            batch=3, 
            imgsz=1024,           
            project=f'{tmp_project_location}', ##copy all the mlflow artifacts training results in this folder to UC
            exist_ok=True,         
        )       
        
        ## Extract experiment/run/name info
        print(f"experiment_id: {experiment_id}")        
        run_id = run.info.run_id
        print(f"run_id: {run_id}")  
        run_name = run.data.tags.get("mlflow.runName")
        print(f"run_name: {run_name}")

        # Save experiment_id, run_id, and run_name in a dictionary
        run_info = {
            "experiment_id": experiment_id,
            "run_id": run_id,
            "run_name": run_name
        }        

finally:
    # Destroy the process group
    dist.destroy_process_group()

In [0]:
# Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100%|██████████| 23/23 [00:10<00:00,  2.15it/s]
#                    all        133       6601       0.84      0.792      0.865      0.512      0.837       0.78      0.855      0.467
# Speed: 0.8ms preprocess, 17.4ms inference, 0.0ms loss, 3.4ms postprocess per image
# Results saved to /tmp/mmt_nuinsseg_yolo_training_results/train

# 2025/03/24 08:32:06 INFO mlflow.system_metrics.system_metrics_monitor: Stopping system metrics monitoring...
# 2025/03/24 08:32:06 INFO mlflow.system_metrics.system_metrics_monitor: Successfully terminated system metrics monitoring!
# 🏃 View run bright-worm-550 at: https://e2-demo-field-eng.cloud.databricks.com/ml/experiments/951504923410753/runs/e3e7cff731f34f34a7723dacacedb9ab
# 🧪 View experiment at: https://e2-demo-field-eng.cloud.databricks.com/ml/experiments/951504923410753
# MLflow: results logged to databricks
# MLflow: disable with 'yolo settings mlflow=False'
# experiment_id: 951504923410753
# run_id: e3e7cff731f34f34a7723dacacedb9ab
# run_name: bright-worm-550

--------------------------

### `[bash] cp` ephemeral `/tmp/{project_name}/train` to Databricks's UC paths 

In [0]:
import datetime
import json
import subprocess

# Get the current date and time
current_datetime = datetime.datetime.now().strftime("%Y%m%d_%H%M")

# Define the source and destination paths
source_path = f"{tmp_project_location}/train"
destination_path = f"{PROJECT_PATH}/training_runs/{current_datetime}_{run_name}_yolo_training_run"

# Use subprocess to execute the cp command
subprocess.run(["cp", "-r", source_path, destination_path], check=True)

# Define the UC path to save the dictionary
uc_path = f"{PROJECT_PATH}/training_runs/{current_datetime}_{run_info['run_name']}_yolo_training_run/run_info.json"

# Write the dictionary to a JSON file
with open(uc_path, 'w') as f:
    json.dump(run_info, f)
print(f"Run information saved to {uc_path}")

--------------------------